In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT php-src -> Test on php-src
GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "php-src"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/gan/php-src"

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Train GAN on minority (vulnerable) class only
# =========================================================
def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Classifier Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on GAN-oversampled source (all-except-php-src)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (php-src)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled php-src
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (php-src Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except php-src -> Test on php-src (GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/6] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/6] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (php-src)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/6] Normalize
    # ----------------------------------------------------------
    print("\n[2/6] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/6] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/6] Training GAN on minority class...")
    minority_emb = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    generator = train_gan(
        minority_emb,
        noise_dim=GAN_NOISE_DIM,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        lr=GAN_LR
    )

    synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
    synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

    train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
    train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/6] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/6] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_gan, train_lbl_gan,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_gan, train_lbl_gan, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/6] Metrics
    # ----------------------------------------------------------
    print("\n[5/6] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (php-src Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [6/6] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "gan",
        "train"            : "all_except_php-src",
        "test"             : "php-src",
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_gan"      : int(len(train_lbl_gan)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_gan" : int(train_lbl_gan.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "n_synthetic_generated": int(n_to_generate),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "gan_php-src.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_gan_php-src.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT Android -> Test on Android
GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "Android"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/gan/Android"

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Train GAN on minority (vulnerable) class only
# =========================================================
def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Classifier Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on GAN-oversampled source (all-except-Android)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (Android)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled Android
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (Android Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except Android -> Test on Android (GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/6] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/6] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (Android)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/6] Normalize
    # ----------------------------------------------------------
    print("\n[2/6] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/6] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/6] Training GAN on minority class...")
    minority_emb = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    generator = train_gan(
        minority_emb,
        noise_dim=GAN_NOISE_DIM,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        lr=GAN_LR
    )

    synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
    synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

    train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
    train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/6] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/6] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_gan, train_lbl_gan,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_gan, train_lbl_gan, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/6] Metrics
    # ----------------------------------------------------------
    print("\n[5/6] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (Android Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [6/6] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "gan",
        "train"            : "all_except_Android",
        "test"             : "Android",
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_gan"      : int(len(train_lbl_gan)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_gan" : int(train_lbl_gan.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "n_synthetic_generated": int(n_to_generate),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "gan_Android.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_gan_Android.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT openssl -> Test on openssl
GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "openssl"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/gan/openssl"

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Train GAN on minority (vulnerable) class only
# =========================================================
def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Classifier Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on GAN-oversampled source (all-except-openssl)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (openssl)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled openssl
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (openssl Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except openssl -> Test on openssl (GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/6] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/6] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (openssl)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/6] Normalize
    # ----------------------------------------------------------
    print("\n[2/6] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/6] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/6] Training GAN on minority class...")
    minority_emb = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    generator = train_gan(
        minority_emb,
        noise_dim=GAN_NOISE_DIM,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        lr=GAN_LR
    )

    synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
    synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

    train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
    train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/6] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/6] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_gan, train_lbl_gan,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_gan, train_lbl_gan, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/6] Metrics
    # ----------------------------------------------------------
    print("\n[5/6] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (openssl Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [6/6] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "gan",
        "train"            : "all_except_openssl",
        "test"             : "openssl",
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_gan"      : int(len(train_lbl_gan)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_gan" : int(train_lbl_gan.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "n_synthetic_generated": int(n_to_generate),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "gan_openssl.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_gan_openssl.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT ImageMagick -> Test on ImageMagick
GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "ImageMagick"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/gan/ImageMagick"

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Train GAN on minority (vulnerable) class only
# =========================================================
def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Classifier Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on GAN-oversampled source (all-except-ImageMagick)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (ImageMagick)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled ImageMagick
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (ImageMagick Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except ImageMagick -> Test on ImageMagick (GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/6] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/6] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (ImageMagick)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/6] Normalize
    # ----------------------------------------------------------
    print("\n[2/6] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/6] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/6] Training GAN on minority class...")
    minority_emb = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    generator = train_gan(
        minority_emb,
        noise_dim=GAN_NOISE_DIM,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        lr=GAN_LR
    )

    synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
    synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

    train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
    train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/6] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/6] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_gan, train_lbl_gan,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_gan, train_lbl_gan, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/6] Metrics
    # ----------------------------------------------------------
    print("\n[5/6] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (ImageMagick Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [6/6] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "gan",
        "train"            : "all_except_ImageMagick",
        "test"             : "ImageMagick",
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_gan"      : int(len(train_lbl_gan)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_gan" : int(train_lbl_gan.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "n_synthetic_generated": int(n_to_generate),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "gan_ImageMagick.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_gan_ImageMagick.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT tensorflow -> Test on tensorflow
GAN-based Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "tensorflow"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/gan/tensorflow"

GAN_NOISE_DIM  = 32
GAN_EPOCHS     = 300
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# GAN: Generator and Discriminator
# =========================================================
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Train GAN on minority (vulnerable) class only
# =========================================================
def train_gan(minority_data, noise_dim, epochs, batch_size, lr):
    input_dim = minority_data.shape[1]
    generator     = Generator(noise_dim, input_dim).to(DEVICE)
    discriminator = Discriminator(input_dim).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_t = torch.tensor(minority_data, dtype=torch.float32)
    dataset    = TensorDataset(minority_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training GAN", leave=False):
        for (real_batch,) in dataloader:
            real_batch = real_batch.to(DEVICE)
            bsz = real_batch.size(0)

            real_labels = torch.ones(bsz, 1).to(DEVICE)
            fake_labels = torch.zeros(bsz, 1).to(DEVICE)

            # --- Train Discriminator ---
            opt_d.zero_grad()
            real_pred = discriminator(real_batch)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            opt_d.step()

            # --- Train Generator ---
            opt_g.zero_grad()
            z = torch.randn(bsz, noise_dim).to(DEVICE)
            fake_batch = generator(z)
            fake_pred = discriminator(fake_batch)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            opt_g.step()

        if (epoch + 1) % 50 == 0:
            print(f"      GAN Epoch {epoch+1}/{epochs} - D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

    return generator


def generate_synthetic_samples(generator, num_samples, noise_dim):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, noise_dim).to(DEVICE)
        synthetic = generator(z).cpu().numpy()
    return synthetic.astype(np.float32)


# =========================================================
# Classifier Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on GAN-oversampled source (all-except-tensorflow)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (tensorflow)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled tensorflow
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (tensorflow Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except tensorflow -> Test on tensorflow (GAN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/6] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/6] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (tensorflow)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/6] Normalize
    # ----------------------------------------------------------
    print("\n[2/6] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/6] Train GAN on minority (vulnerable) class
    # ----------------------------------------------------------
    print("\n[3/6] Training GAN on minority class...")
    minority_emb = train_emb[train_lbl == 1]
    majority_count = int((train_lbl == 0).sum())
    minority_count = int((train_lbl == 1).sum())
    n_to_generate  = majority_count - minority_count

    print(f"      Minority (vulnerable) samples : {minority_count}")
    print(f"      Majority (benign) samples     : {majority_count}")
    print(f"      Synthetic samples to generate : {n_to_generate}")

    generator = train_gan(
        minority_emb,
        noise_dim=GAN_NOISE_DIM,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        lr=GAN_LR
    )

    synthetic_emb = generate_synthetic_samples(generator, n_to_generate, GAN_NOISE_DIM)
    synthetic_lbl = np.ones(n_to_generate, dtype=np.int32)

    train_emb_gan = np.concatenate((train_emb, synthetic_emb)).astype(np.float32)
    train_lbl_gan = np.concatenate((train_lbl, synthetic_lbl)).astype(np.int32)

    print(f"      Before GAN oversampling : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After GAN oversampling  : {train_emb_gan.shape}  "
          f"vuln={train_lbl_gan.sum()}  benign={int((train_lbl_gan == 0).sum())}")

    # ----------------------------------------------------------
    # [4/6] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/6] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_gan, train_lbl_gan,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_gan, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_gan == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_gan, train_lbl_gan, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/6] Metrics
    # ----------------------------------------------------------
    print("\n[5/6] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (tensorflow Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    # ----------------------------------------------------------
    # [6/6] Save results
    # ----------------------------------------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "gan",
        "train"            : "all_except_tensorflow",
        "test"             : "tensorflow",
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_gan"      : int(len(train_lbl_gan)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_gan" : int(train_lbl_gan.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "n_synthetic_generated": int(n_to_generate),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "gan_tensorflow.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_gan_tensorflow.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()